# SentinelFlow — 03: Cloud Layer (Deep Analysis + HITL)

**Stack:** ROCm · vLLM · Qwen2.5  
**Layer:** Tier 3 — cloud/GPU cluster  
**Latency budget:** ≤ 200 ms deep analysis, ≤ 500 ms alerts

### What this notebook does
1. Load escalated events from `correlated_events.json`  
2. **Qwen deep analysis** — full narrative + recommended action per event  
3. Vector embedding + cosine similarity search (in-memory store)  
4. Anomaly scoring — kNN distance from historical baseline  
5. GPU autoscale trigger — signals when queue spikes  
6. **Human-in-the-loop (HITL)** — uncertain events routed to analyst review  
7. Continual learning trigger — submits retraining job on high anomaly rate  
8. Final report + save results

## 0. Load config + correlated events

In [ ]:
import yaml, json, pathlib
cfg = yaml.safe_load(pathlib.Path('config.yml').read_text())
VLLM_BASE_URL = cfg['vllm_base_url']
QWEN_MODEL    = 'qwen'

correlated = json.loads(pathlib.Path('correlated_events.json').read_text())
print(f'Loaded {len(correlated)} correlated events for cloud analysis')

from openai import OpenAI
client = OpenAI(base_url=VLLM_BASE_URL, api_key='not-needed')
print('Qwen client ready')

## 1. Imports + constants

In [ ]:
import time, uuid, json
import numpy as np
from dataclasses import dataclass, field
from typing      import Dict, List, Optional, Tuple
from datetime    import datetime

BUDGET_EMBED_MS       = 50
BUDGET_SEARCH_MS      = 20
BUDGET_ANALYSIS_MS    = 5000     # Qwen deep analysis — slower, worth it
ANOMALY_THRESHOLD     = 0.65     # above this → anomalous
HITL_CONFIDENCE_BAND  = (0.45, 0.70)  # send to analyst if conf in this range
RETRAIN_ANOMALY_RATE  = 0.30     # trigger retrain if >30% events are anomalous
GPU_AUTOSCALE_QUEUE   = 10       # trigger GPU worker scale-up above this

print('✓ Imports done')

## 2. Event embedder

In [ ]:
class EventEmbedder:
    """
    Produces a 128-dim embedding from event metadata.

    Production: use Qwen embeddings via vLLM:
        resp = client.embeddings.create(
            model='qwen',
            input=event_text,
        )
        vector = resp.data[0].embedding

    Or use sentence-transformers on ROCm:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
        vector = model.encode(event_text)
    """
    DIM = 128
    SEV_MAP = {'low': 0.1, 'medium': 0.4, 'high': 0.75, 'critical': 1.0}
    LABEL_CENTROIDS = {}

    def __init__(self):
        rng = np.random.default_rng(99)
        for lbl in ['person','vehicle','bicycle','unknown_object','ship','wildfire_smoke']:
            self.LABEL_CENTROIDS[lbl] = rng.standard_normal(self.DIM).astype(np.float32)

    def embed(self, event: dict) -> np.ndarray:
        t0  = time.perf_counter()
        vec = np.zeros(self.DIM, dtype=np.float32)
        for sub in event.get('source_events', []):
            c   = self.LABEL_CENTROIDS.get(sub.get('label',''), np.zeros(self.DIM))
            vec += c * sub.get('confidence', 0.5)
        # Inject structural features
        vec[0] = self.SEV_MAP.get(event.get('severity','low'), 0.1)
        vec[1] = min(len(event.get('involved_sources', [])) / 8, 1.0)
        vec[2] = event.get('avg_confidence', 0.5)
        norm   = np.linalg.norm(vec)
        vec    = vec / (norm + 1e-9)
        ms     = (time.perf_counter() - t0) * 1000
        return vec

embedder = EventEmbedder()
print('EventEmbedder ready (128-dim)')

## 3. Vector store + anomaly scorer

In [ ]:
class VectorStore:
    """
    In-memory cosine similarity store.
    Production: replace with Milvus or pgvector on ROCm:
        from pymilvus import connections, Collection
        connections.connect(host='localhost', port=19530)
    """
    def __init__(self, max_size: int = 100_000):
        self._ids  : List[str]          = []
        self._vecs : List[np.ndarray]   = []
        self._meta : List[dict]         = []
        self._max  = max_size

    def add(self, event_id: str, vec: np.ndarray, meta: dict = None) -> None:
        if len(self._ids) >= self._max:
            self._ids.pop(0); self._vecs.pop(0); self._meta.pop(0)
        self._ids.append(event_id)
        self._vecs.append(vec / (np.linalg.norm(vec) + 1e-9))
        self._meta.append(meta or {})

    def search(self, query: np.ndarray, top_k: int = 5) -> List[Tuple[str, float, dict]]:
        t0 = time.perf_counter()
        if not self._vecs:
            return []
        q    = query / (np.linalg.norm(query) + 1e-9)
        mat  = np.stack(self._vecs)               # N x dim
        sims = mat @ q                             # cosine similarities
        top  = np.argsort(sims)[::-1][:top_k]
        ms   = (time.perf_counter() - t0) * 1000
        return [(self._ids[i], float(sims[i]), self._meta[i]) for i in top]

    def __len__(self): return len(self._ids)


class AnomalyScorer:
    """kNN-distance anomaly scoring."""
    def __init__(self, store: VectorStore, top_k: int = 5):
        self.store = store
        self.top_k = top_k

    def score(self, vec: np.ndarray) -> float:
        neighbours = self.store.search(vec, self.top_k)
        if not neighbours:
            return 0.85   # no history → treat as moderately anomalous
        avg_sim = sum(sim for _, sim, _ in neighbours) / len(neighbours)
        return round(float(np.clip((1.0 - avg_sim) / 2.0, 0.0, 1.0)), 4)


vector_store = VectorStore()
anomaly_scorer = AnomalyScorer(vector_store)

# Seed with 20 synthetic normal-event embeddings for baseline
rng = np.random.default_rng(1)
for _ in range(20):
    v = rng.standard_normal(128).astype(np.float32)
    v[0] = 0.1   # low severity baseline
    vector_store.add(str(uuid.uuid4()), v, {'baseline': True})

print(f'VectorStore seeded with {len(vector_store)} baseline embeddings')

## 4. HITL router

In [ ]:
class HITLRouter:
    """
    Routes uncertain events to human analysts.
    Analyst labels feed back into the fine-tuning pipeline.

    Production: publish to an analyst review queue (Kafka topic, Slack thread,
    or Label Studio task via the Label Studio SDK).
    """
    def __init__(self, conf_low: float = 0.45, conf_high: float = 0.70):
        self.conf_low  = conf_low
        self.conf_high = conf_high
        self._pending  : List[dict] = []
        self._labels   : List[dict] = []

    def needs_review(self, event: dict, anomaly_score: float) -> bool:
        avg_conf = event.get('avg_confidence', 1.0)
        in_band  = self.conf_low <= avg_conf <= self.conf_high
        return in_band or (anomaly_score >= ANOMALY_THRESHOLD * 1.3)

    def send_to_analyst(self, event: dict, analysis: dict) -> str:
        """Queue event for analyst review. Returns review_id."""
        review_id = str(uuid.uuid4())[:8]
        self._pending.append({
            'review_id'    : review_id,
            'event_type'   : event.get('event_type'),
            'severity'     : event.get('severity'),
            'avg_confidence': event.get('avg_confidence'),
            'anomaly_score': analysis.get('anomaly_score'),
            'summary'      : analysis.get('summary'),
            'queued_at'    : datetime.utcnow().isoformat(),
        })
        print(f'  📋 HITL: event queued for analyst review  [review_id={review_id}]')
        return review_id

    def simulate_analyst_label(self, review_id: str,
                               label: str, correct: bool) -> dict:
        """Simulate an analyst providing a label (for demo)."""
        result = {'review_id': review_id, 'analyst_label': label,
                  'correct': correct, 'labelled_at': datetime.utcnow().isoformat()}
        self._labels.append(result)
        return result

    @property
    def pending_count(self): return len(self._pending)

    @property
    def labels(self): return list(self._labels)

hitl = HITLRouter()
print('HITLRouter ready')

## 5. GPU autoscale trigger

In [ ]:
class GPUAutoscaler:
    """
    Monitors cloud analysis queue and triggers GPU worker scale-up.

    Production:
        import boto3
        asg = boto3.client('autoscaling')
        asg.set_desired_capacity(AutoScalingGroupName='sentinelflow-gpu',
                                  DesiredCapacity=target)
    Or Kubernetes:
        kubectl scale deployment sentinelflow-gpu --replicas=N
    """
    def __init__(self, threshold: int = GPU_AUTOSCALE_QUEUE):
        self.threshold     = threshold
        self._current_gpus = 1
        self._events       : List[dict] = []

    def check_and_scale(self, queue_depth: int) -> int:
        target = max(1, -(-queue_depth // self.threshold))  # ceiling div
        if target != self._current_gpus:
            direction = '⬆ SCALE UP' if target > self._current_gpus else '⬇ SCALE DOWN'
            print(f'  🖥  GPU AUTOSCALE: {direction} '
                  f'{self._current_gpus} → {target} workers '
                  f'(queue depth={queue_depth})')
            self._events.append({'time': datetime.utcnow().isoformat(),
                                  'from': self._current_gpus, 'to': target})
            self._current_gpus = target
        return target

autoscaler = GPUAutoscaler()
print('GPUAutoscaler ready')

## 6. Qwen deep analyser

In [ ]:
QWEN_CLOUD_SYSTEM = """\
You are the Cloud Analyser Agent for SentinelFlow — an intelligent surveillance pipeline.
You receive a single escalated event that has already passed edge filtering and fog correlation.
Provide a thorough analysis.

Respond ONLY with valid JSON:
{
  "summary": "<2-3 sentence narrative of what happened and why it matters>",
  "root_cause": "<1 sentence>",
  "recommended_action": "<specific, actionable instruction>",
  "confidence_assessment": "<1 sentence on detection reliability>",
  "retrain_flag": true/false,
  "retrain_reason": "<1 sentence or null>",
  "anomaly_explanation": "<why this is or is not anomalous>"
}
"""

class QwenDeepAnalyser:
    def __init__(self, model: str = QWEN_MODEL):
        self.model = model

    def analyse(self, event: dict, anomaly_score: float,
                similar_ids: List[str]) -> dict:
        prompt = (
            f'Event type    : {event.get("event_type")}\n'
            f'Severity      : {event.get("severity")}\n'
            f'Sources       : {event.get("involved_sources")}\n'
            f'Avg confidence: {event.get("avg_confidence"):.3f}\n'
            f'Anomaly score : {anomaly_score:.4f}\n'
            f'Similar past events found: {len(similar_ids)}\n'
            f'Rule matched  : {event.get("rule_matched")}\n'
            f'Description   : {event.get("description")}\n'
            f'Qwen fog note : {event.get("qwen_reasoning", "none")}\n'
            'Perform full deep analysis.'
        )
        t0 = time.perf_counter()
        try:
            resp = client.chat.completions.create(
                model=self.model,
                messages=[
                    {'role': 'system', 'content': QWEN_CLOUD_SYSTEM},
                    {'role': 'user',   'content': prompt},
                ],
                max_tokens=512,
                temperature=0.1,
            )
            raw = resp.choices[0].message.content.strip()
            if raw.startswith('```'):
                raw = '\n'.join(l for l in raw.split('\n')
                                if not l.strip().startswith('```')).strip()
            result = json.loads(raw)
        except Exception as e:
            result = {'summary': f'Analysis failed: {e}',
                      'recommended_action': 'Manual review required',
                      'retrain_flag': False, 'retrain_reason': None,
                      'anomaly_explanation': '', 'root_cause': '',
                      'confidence_assessment': ''}
        ms = (time.perf_counter() - t0) * 1000
        if ms > BUDGET_ANALYSIS_MS:
            print(f'  ⚠ DeepAnalysis exceeded budget: {ms:.0f}ms')
        return result

analyser = QwenDeepAnalyser()
print('QwenDeepAnalyser ready')

## 7. Full cloud pipeline — run all escalated events

In [ ]:
final_results = []
hitl_reviews  = []
retrain_flags = []

# GPU autoscale check
autoscaler.check_and_scale(len(correlated))

print(f'Analysing {len(correlated)} escalated events...\n')

for i, event in enumerate(correlated):
    sev_icon = {'low':'🔵','medium':'🟡','high':'🟠','critical':'🔴'}.get(
        event.get('severity','low'),'⚪')
    print(f'── Event {i+1}/{len(correlated)} {sev_icon} '
          f'{event.get("event_type")} ──────────────────────')

    # Step 1: Embed
    vec = embedder.embed(event)

    # Step 2: Search similar past events
    similar = vector_store.search(vec, top_k=5)
    sim_ids = [s[0] for s in similar]

    # Step 3: Anomaly score
    a_score = anomaly_scorer.score(vec)
    a_flag  = '⚠ ANOMALY' if a_score >= ANOMALY_THRESHOLD else '  normal '
    print(f'  Anomaly score: {a_score:.4f}  {a_flag}')
    print(f'  Similar past : {len(similar)} found  '
          f'top_sim={similar[0][1]:.3f}' if similar else '  Similar past: none')

    # Step 4: Store embedding for future queries
    vector_store.add(event.get('correlation_id', str(uuid.uuid4())), vec,
                     {'event_type': event.get('event_type'),
                      'severity'  : event.get('severity')})

    # Step 5: Qwen deep analysis
    analysis = analyser.analyse(event, a_score, sim_ids)
    analysis['anomaly_score'] = a_score
    analysis['similar_ids']   = sim_ids
    analysis['analysis_id']   = str(uuid.uuid4())

    print(f'  Summary: {analysis.get("summary","")[:120]}')
    print(f'  Action : {analysis.get("recommended_action","")[:100]}')
    if analysis.get('retrain_flag'):
        print(f'  🔁 RETRAIN flagged: {analysis.get("retrain_reason","")}' )
        retrain_flags.append(analysis)

    # Step 6: HITL routing
    if hitl.needs_review(event, a_score):
        review_id = hitl.send_to_analyst(event, analysis)
        # Simulate analyst labelling (demo)
        label = hitl.simulate_analyst_label(review_id, event.get('event_type','?'), True)
        hitl_reviews.append(label)

    final_results.append({
        'event'   : event,
        'analysis': analysis,
    })
    print()

# Continual learning trigger
anomaly_rate = sum(1 for r in final_results
                   if r['analysis'].get('anomaly_score', 0) >= ANOMALY_THRESHOLD
                  ) / max(len(final_results), 1)

print('═' * 60)
print(f'Events analysed     : {len(final_results)}')
print(f'Anomaly rate        : {anomaly_rate*100:.0f}%  (threshold={RETRAIN_ANOMALY_RATE*100:.0f}%)')
print(f'HITL review pending : {hitl.pending_count}')
print(f'Retrain flagged     : {len(retrain_flags)}')
print(f'Vector store size   : {len(vector_store)}')

if anomaly_rate > RETRAIN_ANOMALY_RATE or retrain_flags:
    print('\n🔁 CONTINUAL LEARNING TRIGGER: anomaly rate exceeded threshold')
    print('   → Submit fine-tuning job with HITL-labelled frames')
    print(f'   → {len(hitl.labels)} analyst labels available for training')

## 8. Final pipeline report

In [ ]:
print('\n' + '═'*60)
print('  SENTINELFLOW — END-TO-END PIPELINE REPORT')
print('═'*60)

edge_total = 40   # from notebook 01
edge_events_count = len(json.loads(pathlib.Path('edge_events.json').read_text()))
cloud_count = len(final_results)

print(f"""
  [TIER 1 EDGE]
    Raw frames           : {edge_total}
    Events emitted       : {edge_events_count}  ({100*(1-edge_events_count/edge_total):.0f}% reduced)

  [TIER 2 FOG]
    Events ingested      : {edge_events_count}
    Correlated events    : {len(correlated)}
    Escalated to cloud   : {len(correlated)}

  [TIER 3 CLOUD]
    Events analysed      : {cloud_count}
    Anomalies detected   : {sum(1 for r in final_results if r['analysis'].get('anomaly_score',0)>=ANOMALY_THRESHOLD)}
    HITL reviews pending : {hitl.pending_count}
    Retrain triggered    : {anomaly_rate > RETRAIN_ANOMALY_RATE}
    Vector store size    : {len(vector_store)}

  Total data reduction   : {100*(1-cloud_count/edge_total):.0f}%
  (Only {cloud_count/edge_total*100:.1f}% of raw frames needed deep GPU compute)
""")

## 9. Save final results

In [ ]:
pathlib.Path('cloud_results.json').write_text(
    json.dumps(final_results, default=str, indent=2)
)
pathlib.Path('hitl_labels.json').write_text(
    json.dumps(hitl.labels, default=str, indent=2)
)
print(f'Saved cloud_results.json ({len(final_results)} entries)')
print(f'Saved hitl_labels.json   ({len(hitl.labels)} analyst labels)')
print('\n✓ Pipeline complete — see 04_orchestrator.ipynb for the agentic runner')